# Chinatown Signs: Facade Extraction, OCR, Color, and Multilingual Embeddings

Companion to `README.md`. Read the README first for context on provenance and ethics.

**Pipeline:**
1. Study area from OSM
2. Sample points along street centerlines
3. Pull Street View images with metadata
4. Detect signs (bounding boxes)
5. Extract dominant colors per sign
6. OCR: split into Chinese and English tokens
7. Embed tokens in a multilingual model
8. Reduce to 3D with UMAP
9. Visualize: maps + 3D word cloud

One master DataFrame carries provenance through every step. If you find yourself deriving a value without a `sign_id` or `sample_id` attached, stop and merge it back.

## Setup

Install once (uncomment):

In [ ]:
# !pip install requests pillow numpy pandas geopandas shapely osmnx matplotlib \
#     scikit-learn paddleocr paddlepaddle sentence-transformers umap-learn plotly tqdm

In [ ]:
import os
import io
import json
import uuid
import time
import math
import hashlib
from pathlib import Path

import requests
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, LineString
from PIL import Image
from tqdm import tqdm

import matplotlib.pyplot as plt
import osmnx as ox

pd.set_option('display.max_colwidth', 60)

In [ ]:
# Paths. Keep raw imagery outside the repo; only the derived table gets committed.
ROOT = Path('./data/chinatown_signs')
IMG_DIR = ROOT / 'images'
CROP_DIR = ROOT / 'crops'
IMG_DIR.mkdir(parents=True, exist_ok=True)
CROP_DIR.mkdir(parents=True, exist_ok=True)

TABLE_PATH = ROOT / 'signs_table.parquet'

# Your Google Maps Platform key with Street View Static API enabled.
# Do not commit this. Load from an env var or a local untracked file.
GOOGLE_KEY = os.environ.get('GOOGLE_MAPS_KEY') or ''
assert GOOGLE_KEY, 'Set GOOGLE_MAPS_KEY in your environment before continuing.'

## Step 1: Study area

Manhattan Chinatown, roughly bounded by Grand St (north), Worth St (south), Bowery (east), and Broadway (west). Adjust the bounding box for a different neighborhood.

In [ ]:
# North, South, East, West bounds.
BBOX = (40.7205, 40.7128, -73.9925, -74.0030)

# Streets within the bbox from OSM.
G = ox.graph_from_bbox(*BBOX, network_type='walk', simplify=True)
streets = ox.graph_to_gdfs(G, nodes=False, edges=True)
streets = streets[streets.geometry.type == 'LineString'].to_crs(2263)  # NYC feet
print(f'{len(streets)} street segments')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
streets.plot(ax=ax, linewidth=0.7, color='#333')
ax.set_title('Chinatown street centerlines (NAD83 / NY LI ftUS)')
ax.set_axis_off()

## Step 2: Sample points along centerlines

Walk each centerline at `SAMPLE_INTERVAL_FT`. At each point, generate two viewpoints, one for each side of the street (perpendicular offsets), each with a heading that faces the facade.

This produces the first columns of our master table: `sample_id`, `lat`, `lon`, `heading`.

In [ ]:
SAMPLE_INTERVAL_FT = 80  # ~24 m

def sample_line(line, interval):
    dists = np.arange(0, line.length, interval)
    return [(line.interpolate(d), d, line) for d in dists]

def bearing_at(line, dist):
    a = line.interpolate(max(dist - 5, 0))
    b = line.interpolate(min(dist + 5, line.length))
    dx, dy = b.x - a.x, b.y - a.y
    # bearing in projected coords is not compass; we convert at the end after transforming.
    return math.degrees(math.atan2(dx, dy)) % 360

records = []
for _, row in streets.iterrows():
    line = row.geometry
    for pt, d, ln in sample_line(line, SAMPLE_INTERVAL_FT):
        street_heading = bearing_at(ln, d)
        for side_offset in (+90, -90):
            records.append({
                'sample_id': str(uuid.uuid4()),
                'x_ft': pt.x,
                'y_ft': pt.y,
                'heading': (street_heading + side_offset) % 360,
            })
samples = gpd.GeoDataFrame(
    records,
    geometry=[Point(r['x_ft'], r['y_ft']) for r in records],
    crs=2263,
).to_crs(4326)
samples['lon'] = samples.geometry.x
samples['lat'] = samples.geometry.y
print(f'{len(samples)} viewpoints')
samples.head()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
streets.to_crs(4326).plot(ax=ax, linewidth=0.6, color='#888')
samples.plot(ax=ax, markersize=4, color='#3C4ED6')
ax.set_title(f'Viewpoints ({len(samples)})')
ax.set_axis_off()

## Step 3: Fetch Street View images with metadata

For each viewpoint, we call two endpoints:

- `metadata` first, to get `pano_id` and `date` and to confirm coverage exists. This is free.
- `streetview` if metadata was OK, to get the JPEG.

We cache by a hash of `(lat, lon, heading)` so re-runs do not re-charge.

**Cost warning:** the paid Street View Static call is ~$7 per 1000. This notebook will not run in bulk without the API key. To develop the pipeline, set `LIMIT` to a small number.

In [ ]:
SV_META = 'https://maps.googleapis.com/maps/api/streetview/metadata'
SV_IMG  = 'https://maps.googleapis.com/maps/api/streetview'
SIZE = '640x400'
FOV = 80
PITCH = 5
LIMIT = 20  # set to len(samples) for a real run

def sv_key(lat, lon, heading):
    return hashlib.md5(f'{lat:.6f},{lon:.6f},{heading:.1f}'.encode()).hexdigest()

def fetch_streetview(lat, lon, heading):
    key = sv_key(lat, lon, heading)
    img_path = IMG_DIR / f'{key}.jpg'
    meta_path = IMG_DIR / f'{key}.json'

    if img_path.exists() and meta_path.exists():
        meta = json.loads(meta_path.read_text())
        return meta, str(img_path)

    params = {
        'location': f'{lat},{lon}',
        'heading': heading,
        'radius': 30,
        'source': 'outdoor',
        'key': GOOGLE_KEY,
    }
    meta = requests.get(SV_META, params=params).json()
    if meta.get('status') != 'OK':
        return meta, None

    r = requests.get(SV_IMG, params={**params, 'size': SIZE, 'fov': FOV, 'pitch': PITCH})
    if r.status_code != 200:
        return meta, None
    img_path.write_bytes(r.content)
    meta_path.write_text(json.dumps(meta))
    return meta, str(img_path)

In [ ]:
batch = samples.head(LIMIT).copy()
results = []
for _, row in tqdm(batch.iterrows(), total=len(batch)):
    meta, path = fetch_streetview(row.lat, row.lon, row.heading)
    results.append({
        'sample_id': row.sample_id,
        'pano_id': meta.get('pano_id'),
        'capture_date': meta.get('date'),
        'image_path': path,
        'status': meta.get('status'),
    })

images = pd.DataFrame(results)
records_df = batch.merge(images, on='sample_id')
records_df = records_df.dropna(subset=['image_path']).reset_index(drop=True)
print(f'{len(records_df)} images fetched successfully')
records_df.head()

## Step 4: Detect signs

For the walkthrough we use PaddleOCR's detection stage as a cheap sign locator: any text region is treated as a sign candidate. This works well for storefront signage because storefronts are text-heavy.

For a more selective run, swap in a foundation model:
- Google Vision API `objectLocalization` with `Signage` label filter
- Or a YOLO fine-tuned on the OpenStreetSigns dataset

The output is `signs_df`: one row per detected sign, joined back to `records_df` by `sample_id`.

In [ ]:
from paddleocr import PaddleOCR

# ch model handles both Chinese and English.
ocr = PaddleOCR(use_angle_cls=True, lang='ch', show_log=False)

def detect_signs(image_path):
    result = ocr.ocr(image_path, cls=True)
    if not result or not result[0]:
        return []
    signs = []
    for line in result[0]:
        bbox, (text, conf) = line
        xs = [p[0] for p in bbox]
        ys = [p[1] for p in bbox]
        signs.append({
            'bbox': [min(xs), min(ys), max(xs) - min(xs), max(ys) - min(ys)],
            'text': text,
            'conf': float(conf),
        })
    return signs

In [ ]:
sign_rows = []
for _, row in tqdm(records_df.iterrows(), total=len(records_df)):
    for det in detect_signs(row.image_path):
        # Crop and save. This ties the sign to a specific rectangle of a specific image.
        img = Image.open(row.image_path)
        x, y, w, h = det['bbox']
        pad = 4
        crop = img.crop((max(0, x-pad), max(0, y-pad), x+w+pad, y+h+pad))
        sign_id = str(uuid.uuid4())
        crop_path = CROP_DIR / f'{sign_id}.jpg'
        crop.save(crop_path)

        sign_rows.append({
            **row.to_dict(),
            'sign_id': sign_id,
            'bbox': det['bbox'],
            'sign_crop_path': str(crop_path),
            'ocr_text': det['text'],
            'ocr_conf': det['conf'],
        })

signs_df = pd.DataFrame(sign_rows)
print(f'{len(signs_df)} signs across {signs_df.sample_id.nunique()} images')
signs_df[['sample_id', 'sign_id', 'ocr_text', 'ocr_conf']].head(10)

## Step 5: Dominant colors per sign

K-means on the RGB pixels of each crop. Cluster centers are the palette; cluster sizes are the weights. We save the top three and pick the largest as `dominant_color_hex`.

In [ ]:
from sklearn.cluster import KMeans

def dominant_colors(image_path, k=3):
    img = Image.open(image_path).convert('RGB')
    # Downsample for speed; sign crops are already small.
    img.thumbnail((80, 80))
    pixels = np.array(img).reshape(-1, 3)
    if len(pixels) < k:
        return []
    km = KMeans(n_clusters=k, n_init=4, random_state=0).fit(pixels)
    counts = np.bincount(km.labels_, minlength=k)
    order = np.argsort(-counts)
    return [
        (int(km.cluster_centers_[i][0]),
         int(km.cluster_centers_[i][1]),
         int(km.cluster_centers_[i][2]),
         float(counts[i] / counts.sum()))
        for i in order
    ]

def rgb_to_hex(rgb):
    return '#{:02x}{:02x}{:02x}'.format(*rgb[:3])

signs_df['colors'] = signs_df.sign_crop_path.apply(dominant_colors)
signs_df['dominant_color_hex'] = signs_df.colors.apply(
    lambda cs: rgb_to_hex(cs[0]) if cs else '#000000'
)
signs_df[['sign_id', 'ocr_text', 'dominant_color_hex']].head()

## Step 6: Split OCR text into Chinese and English

Signs frequently mix scripts. We keep the raw string and add split columns. The regex ranges cover CJK Unified Ideographs, CJK Extension A, and common punctuation.

In [ ]:
import re

CJK = re.compile(r'[\u4e00-\u9fff\u3400-\u4dbf]+')
LATIN_WORD = re.compile(r"[A-Za-z][A-Za-z'&\-]*")

def split_text(t):
    if not isinstance(t, str):
        return [], []
    zh = CJK.findall(t)
    en = LATIN_WORD.findall(t)
    return zh, en

signs_df[['chinese_chunks', 'english_words']] = signs_df.ocr_text.apply(
    lambda t: pd.Series(split_text(t))
)

def label_lang(row):
    zh, en = row.chinese_chunks, row.english_words
    if zh and en: return 'mixed'
    if zh:        return 'zh'
    if en:        return 'en'
    return 'other'

signs_df['ocr_language'] = signs_df.apply(label_lang, axis=1)
signs_df.ocr_language.value_counts()

## Step 7: Multilingual embeddings

`paraphrase-multilingual-MiniLM-L12-v2` maps text in 50+ languages into a shared 384-dim space. Chinese and English translations of the same phrase land close together. This is what makes the 3D word cloud legible across scripts.

We embed at the **token** level: each Chinese chunk and each English word gets its own row in a long table with a `sign_id` foreign key. That way the visualization can color by originating sign, sample, or neighborhood block.

In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
model = SentenceTransformer(MODEL_NAME)

# Flatten to one row per token, keeping sign_id and language.
token_rows = []
for _, row in signs_df.iterrows():
    for chunk in row.chinese_chunks:
        token_rows.append({'sign_id': row.sign_id, 'token': chunk, 'lang': 'zh'})
    for word in row.english_words:
        token_rows.append({'sign_id': row.sign_id, 'token': word, 'lang': 'en'})

tokens_df = pd.DataFrame(token_rows).drop_duplicates(subset=['sign_id', 'token'])
print(f'{len(tokens_df)} tokens')

if len(tokens_df):
    embeddings = model.encode(tokens_df.token.tolist(), show_progress_bar=True, batch_size=64)
    tokens_df['embedding'] = list(embeddings)
    print('Embedding shape:', embeddings.shape)

## Step 8: Reduce to 3D with UMAP

UMAP preserves local neighborhoods better than PCA and produces layouts that read like a word cloud. `n_neighbors` controls how much local vs global structure is kept. Small values (5 to 15) produce tighter local clusters; large values (50+) preserve global topology at the cost of local density.

**Important:** after UMAP, coordinates are no longer metric. Do not compute cosine similarity on `x3d, y3d, z3d`. If you need similarity, use the 384-dim `embedding`.

In [ ]:
import umap

if len(tokens_df) >= 5:
    reducer = umap.UMAP(n_components=3, n_neighbors=15, min_dist=0.1, random_state=0, metric='cosine')
    coords_3d = reducer.fit_transform(np.stack(tokens_df.embedding.values))
    tokens_df[['x3d', 'y3d', 'z3d']] = coords_3d

tokens_df.head()

## Step 9a: Map dominant colors and language mix

Two quick maps that connect the semantic dataset back to the ground:

- Signs colored by their `dominant_color_hex`
- Signs colored by `ocr_language` (zh / en / mixed)

In [ ]:
signs_gdf = gpd.GeoDataFrame(
    signs_df,
    geometry=gpd.points_from_xy(signs_df.lon, signs_df.lat),
    crs=4326,
)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

streets.to_crs(4326).plot(ax=axes[0], color='#ccc', linewidth=0.5)
signs_gdf.plot(ax=axes[0], color=signs_gdf.dominant_color_hex, markersize=25, edgecolor='#333', linewidth=0.3)
axes[0].set_title('Signs by dominant color')
axes[0].set_axis_off()

lang_color = {'zh': '#EE352E', 'en': '#3C4ED6', 'mixed': '#FCCC0A', 'other': '#888'}
streets.to_crs(4326).plot(ax=axes[1], color='#ccc', linewidth=0.5)
signs_gdf.plot(ax=axes[1], color=signs_gdf.ocr_language.map(lang_color), markersize=25, edgecolor='#333', linewidth=0.3)
axes[1].set_title('Signs by language mix')
axes[1].set_axis_off()
plt.tight_layout()

## Step 9b: 3D word cloud in embedding space

Each point is a token. Color by language. Hover to read the token. Neighborhoods in this space are learned semantic relations, not geographic ones.

Things to look for as you rotate:

- **Translation pairs.** "restaurant" and 餐廳 should be close. If they are not, the model may be favoring script over meaning, or your text tokens are too short.
- **Business type clusters.** Bakeries, herbal medicine shops, jewelry stores each form loose clumps.
- **Frequency vs meaning.** Very common tokens ("co", "inc", 有限公司) drift toward the center because they co-occur with everything.
- **Noise.** OCR errors show up as isolated singletons. That is a signal to lower the OCR confidence threshold or manually filter.

In [ ]:
import plotly.express as px

if 'x3d' in tokens_df.columns:
    fig = px.scatter_3d(
        tokens_df,
        x='x3d', y='y3d', z='z3d',
        color='lang',
        hover_data=['token'],
        text='token',
        color_discrete_map={'zh': '#EE352E', 'en': '#3C4ED6'},
    )
    fig.update_traces(marker=dict(size=4), textfont=dict(size=9))
    fig.update_layout(
        scene=dict(xaxis_title='', yaxis_title='', zaxis_title=''),
        title=f'UMAP-3D of sign tokens ({MODEL_NAME.split("/")[-1]})',
    )
    fig.show()

## Sanity check: nearest neighbors in the full-dim space

The UMAP plot is qualitative. To confirm that translations really are neighbors in the embedding, query the original 384-dim vectors.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def nearest(token, n=8):
    if token not in tokens_df.token.values:
        return None
    idx = tokens_df.token.tolist().index(token)
    vecs = np.stack(tokens_df.embedding.values)
    sims = cosine_similarity(vecs[idx:idx+1], vecs)[0]
    top = np.argsort(-sims)[1:n+1]
    return tokens_df.iloc[top][['token', 'lang']].assign(sim=sims[top])

# Example: try a common token from your data.
# nearest('restaurant')
# nearest('餐廳')

## Persist the tables

Two tables, one file each. Keep them small enough to commit.

In [ ]:
# Signs table: one row per sign, provenance + text + color.
# Drop the embedding-heavy tokens table columns before saving; save it separately.
signs_df_out = signs_df.drop(columns=[c for c in signs_df.columns if c.startswith('_')], errors='ignore')
signs_df_out.to_parquet(TABLE_PATH)

# Tokens table: separately, since embeddings are heavy.
if len(tokens_df):
    tokens_df.to_parquet(ROOT / 'tokens_table.parquet')

print(f'Saved {len(signs_df_out)} signs to {TABLE_PATH}')

## What to try next

- **Swap in a sign detector.** Replace the PaddleOCR-as-detector shortcut with a real sign detector so you catch pictorial signs (dragons, produce, symbols) that OCR misses.
- **Compare neighborhoods.** Rerun with Flushing or Sunset Park bounding boxes. Store both in the same tokens table with a `neighborhood` column and see whether the embedding space differentiates them.
- **Historical.** Query Street View metadata by year (`capture_date`); build a sequence of the same corner over a decade.
- **Color theory.** Group dominant colors into named palettes (Munsell, Berlin & Kay categories) and map at the block level.
- **Web deployment.** Export `signs_gdf.to_file('signs.geojson')` and load it into `tutorials/Web/tutorial-2-maps-scrollytelling/` for an interactive story.

## Reminders

- Do not commit the `data/chinatown_signs/images/` directory.
- Do not publish raw Google Street View images. Derived data is fine.
- Blur faces if any people appear in the crops you display.
- Pin the embedding model version in your writeup so results are reproducible.